# Estimate Transformer Parameters, Memory and FLOPs — Solution

**Difficulty**: 🟡 Medium

### Problem Statement

"Can we fully fine-tune a 7B model on 8×80 GB GPUs?" "How much KV cache does a 32k-token
context cost?" "How many GPU-hours does pre-training take?" Research-engineer and ML-infra
interviews expect you to answer these in a few minutes from a model config, and real capacity
planning starts the same way. Everything reduces to four counts:

- **Parameters** $N$ — from the architecture (embedding, attention, MLP, norms, LM head).
- **Training memory** — bytes per parameter for weights, gradients and optimizer state
  (activations come on top and depend on batch and sequence length).
- **KV cache** — the keys and values every layer keeps for every past token during generation.
- **Training compute** — $C \approx 6ND$ FLOPs for $D$ training tokens: $2N$ per token forward
  (one multiply-add per parameter) and $4N$ backward (gradients with respect to both the
  activations and the weights).

You will implement each as a small function and check it against real PyTorch objects: a model
built from `nn` modules, real K/V tensors, real optimizer state, and PyTorch's FLOP counter.

---

### Requirements

1. **`transformer_params(vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff,
   tied_embeddings=True, gated_mlp=True, bias=False, norm="rmsnorm") -> int`** — the exact
   parameter count of a Llama-style decoder-only transformer:
   - `head_dim = d_model // n_heads` (assume `d_model % n_heads == 0`, `n_heads % n_kv_heads == 0`).
   - Token embedding: `vocab_size × d_model`. No positional-embedding parameters (RoPE has none).
   - Each of the `n_layers` blocks:
     - a norm before attention; `q_proj`: `d_model → n_heads·head_dim`; `k_proj` and `v_proj`:
       `d_model → n_kv_heads·head_dim` (grouped-query attention — MHA when `n_kv_heads == n_heads`,
       MQA when it is 1); `o_proj`: `n_heads·head_dim → d_model`;
     - a norm before the MLP; gated (SwiGLU) MLP: `gate_proj` and `up_proj` `d_model → d_ff`,
       `down_proj` `d_ff → d_model`; plain MLP (`gated_mlp=False`): only `up_proj` and `down_proj`.
   - `bias=True`: every `Linear` listed above has a bias (one value per output feature).
   - Norms: `"rmsnorm"` has a weight only (`d_model` values); `"layernorm"` has weight and bias
     (`2·d_model`). Anything else → `ValueError`.
   - A final norm after the last block, then the LM head `d_model → vocab_size` **without bias**,
     which shares the embedding matrix when `tied_embeddings=True` (count it once).
2. **`training_memory_gb(n_params, precision="bf16-mixed", optimizer="adamw") -> float`** — decimal
   gigabytes (bytes / 1e9) of weights + gradients + optimizer state, excluding activations and
   temporary buffers, using this standard accounting (bytes per parameter):

   | `precision` | weights | gradients | fp32 master weights |
   |---|---|---|---|
   | `"fp32"` | 4 | 4 | – |
   | `"bf16-mixed"` | 2 (bf16) | 2 (bf16) | 4 |
   | `"fp16-mixed"` | 2 (fp16) | 2 (fp16) | 4 |

   | `optimizer` | fp32 state per parameter | bytes |
   |---|---|---|
   | `"adamw"`, `"adam"` | first moment $m$, second moment $v$ | 8 |
   | `"sgd-momentum"` | momentum buffer | 4 |
   | `"sgd"` | none | 0 |

   So AdamW costs **16 bytes/parameter** in both fp32 and mixed precision (the classic
   "2 + 2 + 12" of the ZeRO paper). Unknown `precision` or `optimizer` → `ValueError`.
3. **`kv_cache_bytes(batch, seq_len, n_layers, n_kv_heads, head_dim, bytes_per_elem=2) -> int`** —
   bytes needed to cache K and V for every layer and token: one K and one V tensor of shape
   `(batch, n_kv_heads, seq_len, head_dim)` per layer.
4. **`training_flops(n_params, n_tokens) -> float`** — the standard estimate $6\,N\,D$.

---

### Constraints

- ✅ Plain Python arithmetic — these are formulas you should be able to write on a whiteboard.
- ❌ Don't instantiate a model inside `transformer_params` (the tests build one to check *you*,
  and a real 8B model would not fit in memory anyway).
- ❌ No `torch.utils.flop_counter` inside `training_flops`.

---

<details>
  <summary>💡 Hint</summary>

  - Write the count of one `nn.Linear(fan_in, fan_out, bias=b)` as a helper first.
  - With GQA, which projections shrink? Which ones keep `d_model` outputs?
  - In PyTorch, tying is `lm_head.weight = embed.weight`; `model.parameters()` yields a shared
    tensor once. How many vocabulary-sized matrices does an untied model have?
  - For memory, list *what tensors exist* during a mixed-precision step and how many bytes each
    element takes. Which of them does `torch.optim.AdamW` create?
  - For FLOPs: a matmul `(T, a) @ (a, b)` costs $2Tab$. In backward, which two matmuls does each
    `Linear` run?

</details>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.flop_counter import FlopCounterMode

In [ ]:
# Real model configs (from the published Hugging Face config.json files)
LLAMA2_7B = dict(vocab_size=32000, d_model=4096, n_layers=32, n_heads=32, n_kv_heads=32,
                 d_ff=11008, tied_embeddings=False)
LLAMA3_8B = dict(vocab_size=128256, d_model=4096, n_layers=32, n_heads=32, n_kv_heads=8,
                 d_ff=14336, tied_embeddings=False)


class Block(nn.Module):
    """One Llama-style decoder block — parameters only (no forward needed to count them)."""

    def __init__(self, d, n_heads, n_kv, d_ff, gated=True, bias=False, norm="rmsnorm"):
        super().__init__()
        hd = d // n_heads
        Norm = nn.RMSNorm if norm == "rmsnorm" else nn.LayerNorm
        self.attn_norm, self.mlp_norm = Norm(d), Norm(d)
        self.q_proj = nn.Linear(d, n_heads * hd, bias=bias)
        self.k_proj = nn.Linear(d, n_kv * hd, bias=bias)
        self.v_proj = nn.Linear(d, n_kv * hd, bias=bias)
        self.o_proj = nn.Linear(n_heads * hd, d, bias=bias)
        self.gate_proj = nn.Linear(d, d_ff, bias=bias) if gated else None
        self.up_proj = nn.Linear(d, d_ff, bias=bias)
        self.down_proj = nn.Linear(d_ff, d, bias=bias)


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff,
                 tied_embeddings=True, gated_mlp=True, bias=False, norm="rmsnorm"):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(Block(d_model, n_heads, n_kv_heads, d_ff, gated_mlp, bias, norm)
                                    for _ in range(n_layers))
        self.norm = nn.RMSNorm(d_model) if norm == "rmsnorm" else nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tied_embeddings:
            self.lm_head.weight = self.embed_tokens.weight


def count_real(**cfg):
    """Build the model on the meta device (no memory allocated) and count its parameters."""
    with torch.device("meta"):
        model = Decoder(**cfg)
    return sum(p.numel() for p in model.parameters())   # tied weights are yielded once


print(f"Llama-2-7B, built for real: {count_real(**LLAMA2_7B):,} parameters")

In [ ]:
def transformer_params(vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff,
                       tied_embeddings=True, gated_mlp=True, bias=False, norm="rmsnorm"):
    """Exact parameter count of a Llama-style decoder-only transformer.

    Args:
        vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff: architecture sizes.
        tied_embeddings: LM head shares the token-embedding matrix.
        gated_mlp: SwiGLU MLP with 3 matrices (gate, up, down) instead of 2 (up, down).
        bias: every attention/MLP Linear has a bias (the LM head never does).
        norm: "rmsnorm" (weight) or "layernorm" (weight + bias).

    Returns:
        int — number of parameters.
    """
    if norm not in ("rmsnorm", "layernorm"):
        raise ValueError(f"unknown norm {norm!r}")
    if d_model % n_heads or n_heads % n_kv_heads:
        raise ValueError("need d_model % n_heads == 0 and n_heads % n_kv_heads == 0")
    head_dim = d_model // n_heads

    def linear(fan_in, fan_out):
        return fan_in * fan_out + (fan_out if bias else 0)

    norm_params = d_model * (2 if norm == "layernorm" else 1)
    q_out, kv_out = n_heads * head_dim, n_kv_heads * head_dim
    attention = (linear(d_model, q_out) + 2 * linear(d_model, kv_out)   # q, then k and v
                 + linear(q_out, d_model))                              # o
    mlp = ((2 if gated_mlp else 1) * linear(d_model, d_ff)              # (gate,) up
           + linear(d_ff, d_model))                                     # down
    per_layer = attention + mlp + 2 * norm_params

    embedding = vocab_size * d_model
    lm_head = 0 if tied_embeddings else vocab_size * d_model            # never a bias
    return embedding + n_layers * per_layer + norm_params + lm_head

In [ ]:
# bytes per parameter: (weights, gradients, fp32 master weights)
_PRECISION_BYTES = {
    "fp32": (4, 4, 0),
    "bf16-mixed": (2, 2, 4),
    "fp16-mixed": (2, 2, 4),
}
# fp32 optimizer state per parameter
_OPTIMIZER_BYTES = {
    "adamw": 8,          # exp_avg (m) + exp_avg_sq (v)
    "adam": 8,
    "sgd-momentum": 4,   # momentum_buffer
    "sgd": 0,
}


def training_memory_gb(n_params, precision="bf16-mixed", optimizer="adamw"):
    """Weights + gradients + optimizer state, in decimal GB (bytes / 1e9). No activations.

    Args:
        n_params: number of parameters.
        precision: "fp32", "bf16-mixed" or "fp16-mixed".
        optimizer: "adamw", "adam", "sgd-momentum" or "sgd".

    Returns:
        float — gigabytes.
    """
    if precision not in _PRECISION_BYTES:
        raise ValueError(f"unknown precision {precision!r}; choose from {list(_PRECISION_BYTES)}")
    if optimizer not in _OPTIMIZER_BYTES:
        raise ValueError(f"unknown optimizer {optimizer!r}; choose from {list(_OPTIMIZER_BYTES)}")
    bytes_per_param = sum(_PRECISION_BYTES[precision]) + _OPTIMIZER_BYTES[optimizer]
    return n_params * bytes_per_param / 1e9

In [ ]:
def kv_cache_bytes(batch, seq_len, n_layers, n_kv_heads, head_dim, bytes_per_elem=2):
    """Bytes of the K and V cache for every layer and position.

    Returns:
        int — bytes.
    """
    # 2 = one K and one V tensor of shape (batch, n_kv_heads, seq_len, head_dim) per layer
    return 2 * batch * seq_len * n_layers * n_kv_heads * head_dim * bytes_per_elem

In [ ]:
def training_flops(n_params, n_tokens):
    """Approximate training compute: forward + backward over n_tokens tokens.

    Returns:
        float — FLOPs.
    """
    # forward: one multiply-add (2 FLOPs) per parameter per token;
    # backward: twice that (grad w.r.t. the layer input and grad w.r.t. the weight)
    return 6 * n_params * n_tokens

In [ ]:
# Test 1: parameter counts agree with models built from nn modules
configs = [
    dict(LLAMA2_7B),                                                       # MHA, untied
    dict(LLAMA3_8B),                                                       # GQA (8 KV heads)
    dict(vocab_size=101, d_model=64, n_layers=3, n_heads=8, n_kv_heads=2, d_ff=172),   # tied
    dict(vocab_size=50, d_model=48, n_layers=2, n_heads=6, n_kv_heads=1, d_ff=192,
         gated_mlp=False, bias=True, norm="layernorm"),                    # MQA, GPT-style MLP
]
for cfg in configs:
    got, want = transformer_params(**cfg), count_real(**cfg)
    assert got == want, f"{cfg}: got {got:,}, the real model has {want:,} (off by {got - want:,})"
assert transformer_params(**LLAMA2_7B) == 6_738_415_616
assert transformer_params(**LLAMA3_8B) == 8_030_261_248
print(f"Test 1 passed: Llama-2-7B {transformer_params(**LLAMA2_7B) / 1e9:.2f}B, "
      f"Llama-3-8B {transformer_params(**LLAMA3_8B) / 1e9:.2f}B")

In [ ]:
# Test 2: training memory — the 16 bytes/param rule, and real AdamW state
n = transformer_params(**LLAMA2_7B)
assert abs(training_memory_gb(n, "bf16-mixed", "adamw") - n * 16 / 1e9) < 1e-6
assert abs(training_memory_gb(n, "fp32", "adamw") - n * 16 / 1e9) < 1e-6
assert abs(training_memory_gb(n, "fp32", "sgd") - n * 8 / 1e9) < 1e-6
assert abs(training_memory_gb(n, "bf16-mixed", "sgd-momentum") - n * 12 / 1e9) < 1e-6

lin = nn.Linear(256, 128)                         # fp32 model, AdamW: count the real tensors
opt = torch.optim.AdamW(lin.parameters())
lin(torch.randn(4, 256)).sum().backward()
opt.step()
P = sum(p.numel() for p in lin.parameters())
held = sum(t.numel() * t.element_size() for p in lin.parameters()
           for t in [p, p.grad, *opt.state[p].values()] if t.shape == p.shape)
assert abs(training_memory_gb(P, "fp32", "adamw") - held / 1e9) < 1e-15
try:
    training_memory_gb(n, "int3")
    raise AssertionError("unknown precision should raise ValueError")
except ValueError:
    pass
print(f"Test 2 passed: full fine-tuning Llama-2-7B with AdamW needs "
      f"{training_memory_gb(n):.1f} GB before activations")

In [ ]:
# Test 3: KV cache size == bytes of the K/V tensors a decoder actually allocates
batch, seq, layers, kv_heads, head_dim = 2, 10, 3, 2, 8
cache = [(torch.empty(batch, kv_heads, seq, head_dim, dtype=torch.bfloat16),
          torch.empty(batch, kv_heads, seq, head_dim, dtype=torch.bfloat16)) for _ in range(layers)]
real = sum(k.numel() * k.element_size() + v.numel() * v.element_size() for k, v in cache)
assert kv_cache_bytes(batch, seq, layers, kv_heads, head_dim, bytes_per_elem=2) == real
assert kv_cache_bytes(batch, seq, layers, kv_heads, head_dim, bytes_per_elem=4) == 2 * real
llama2 = kv_cache_bytes(1, 4096, 32, 32, 128)
assert llama2 == 2 ** 31
print(f"Test 3 passed: Llama-2-7B, 4k context, batch 1, bf16: {llama2 / 1e9:.2f} GB of KV cache")

In [ ]:
# Test 4: 6·N·D matches PyTorch's FLOP counter on a forward + backward pass
net = nn.Sequential(nn.Linear(32, 64, bias=False), nn.Linear(64, 48, bias=False),
                    nn.Linear(48, 16, bias=False))
x = torch.randn(20, 32, requires_grad=True)       # 20 tokens
with FlopCounterMode(display=False) as counter:
    net(x).sum().backward()
P = sum(p.numel() for p in net.parameters())
assert training_flops(P, 20) == counter.get_total_flops(), \
    f"6ND = {training_flops(P, 20)}, measured {counter.get_total_flops()}"
print(f"Test 4 passed: {counter.get_total_flops():,} FLOPs measured = 6 x {P} params x 20 tokens")
print(f"Llama-2-7B on 2T tokens: {training_flops(transformer_params(**LLAMA2_7B), 2e12):.2e} FLOPs")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("memory-flops-estimation", transformer_params=transformer_params,
      training_memory_gb=training_memory_gb, kv_cache_bytes=kv_cache_bytes,
      training_flops=training_flops)

### Worked examples (the numbers interviewers expect you to produce)

In [ ]:
cfg = LLAMA2_7B
N = transformer_params(**cfg)
d, V, L, dff = cfg["d_model"], cfg["vocab_size"], cfg["n_layers"], cfg["d_ff"]
print("Llama-2-7B parameter breakdown")
print(f"  embedding        V·d          = {V * d:>13,}")
print(f"  per layer: attn  4·d²         = {4 * d * d:>13,}")
print(f"             MLP   3·d·d_ff     = {3 * d * dff:>13,}")
print(f"             norms 2·d          = {2 * d:>13,}")
print(f"  x {L} layers                   = {L * (4 * d * d + 3 * d * dff + 2 * d):>13,}")
print(f"  final norm + LM head (untied) = {d + V * d:>13,}")
print(f"  total                         = {N:>13,}  (≈ {N / 1e9:.2f}B)")

print("\nTraining state (no activations), decimal GB")
for prec, optim in [("bf16-mixed", "adamw"), ("fp32", "adamw"), ("bf16-mixed", "sgd-momentum")]:
    print(f"  {prec:>10} + {optim:<12} {training_memory_gb(N, prec, optim):7.1f} GB")
print(f"  bf16 weights alone (inference) {2 * N / 1e9:7.1f} GB")

# Can we fully fine-tune it on 8 x 80 GB? Per-GPU memory under ZeRO (Psi = N, K = 12 bytes of
# fp32 master + Adam state, Nd = 8 data-parallel ranks), before activations:
Psi, K, Nd = N, 12, 8
zero = {"DDP (replicated)": (2 + 2 + K) * Psi,
        "ZeRO-1 (shard optimizer)": 2 * Psi + 2 * Psi + K * Psi / Nd,
        "ZeRO-2 (+ shard grads)": 2 * Psi + (2 + K) * Psi / Nd,
        "ZeRO-3 / FSDP (+ shard params)": (2 + 2 + K) * Psi / Nd}
print("\nPer-GPU training state for Llama-2-7B on 8 GPUs")
for name, b in zero.items():
    print(f"  {name:<32} {b / 1e9:6.1f} GB {'(does not fit in 80 GB)' if b > 80e9 else ''}")

print("\nKV cache, batch 1, bf16")
for name, c in [("Llama-2-7B (32 KV heads)", LLAMA2_7B), ("Llama-3-8B (8 KV heads, GQA)", LLAMA3_8B)]:
    hd = c["d_model"] // c["n_heads"]
    per_tok = kv_cache_bytes(1, 1, c["n_layers"], c["n_kv_heads"], hd)
    print(f"  {name:<30} {per_tok / 2**10:5.0f} KiB/token, "
          f"4k ctx {kv_cache_bytes(1, 4096, c['n_layers'], c['n_kv_heads'], hd) / 1e9:5.2f} GB, "
          f"32k ctx {kv_cache_bytes(1, 32768, c['n_layers'], c['n_kv_heads'], hd) / 1e9:5.2f} GB")

# Compute: Llama-2-7B was pre-trained on 2T tokens; Meta reports 184,320 A100-80GB GPU-hours.
C = training_flops(N, 2e12)
gpu_hours = 184_320
achieved = C / (gpu_hours * 3600)
print(f"\nLlama-2-7B pre-training: 6ND = {C:.2e} FLOPs")
print(f"  implied throughput {achieved / 1e12:.0f} TFLOP/s per A100 "
      f"= {achieved / 312e12:.0%} MFU (A100 bf16 dense peak: 312 TFLOP/s)")
attn = 12 * L * d * 4096          # PaLM's per-token attention term at 4k context
print(f"  attention term 12·L·d·s at s=4096 adds {attn / (6 * N):.0%} on top of 6N per token")

### Why it works

**Parameters.** A `Linear(a, b)` holds $ab$ weights (+ $b$ biases). Per layer, attention is
$d\cdot d$ for $q$ and $o$, but only $d\cdot n_{kv} d_h$ for each of $k$ and $v$ — GQA shrinks K/V
projections and the KV cache, not the query side. The SwiGLU MLP has three $d \times d_{ff}$
matrices; Llama picks $d_{ff} \approx \frac{8}{3}d$ (11008 for $d = 4096$) so it costs about the
same as a plain $4d$ MLP with two. Norms are tiny ($d$ or $2d$). Tied embeddings save
$V d$ — 131M parameters for Llama-2 ($V = 32$k) and 525M for Llama-3 ($V = 128$k), which is why
small models usually tie and large ones often don't. A useful approximation for a standard
block is $12 L d^2$ (attention $4d^2$ + MLP $8d^2$) plus embeddings.

**Memory.** Mixed-precision Adam keeps a bf16 working copy of the weights (2 B), bf16 gradients
(2 B), an fp32 master copy that the update is applied to (4 B — bf16 cannot represent small
updates to large weights), and Adam's $m$ and $v$ in fp32 (8 B): 16 B/parameter, the same as
pure fp32 (4 + 4 + 8). Mixed precision saves *activation* memory and compute, not optimizer
memory. The spec checks this against real tensors: after `optimizer.step()`, `AdamW` holds
`exp_avg` and `exp_avg_sq`, `SGD(momentum=…)` holds `momentum_buffer`, plain `SGD` nothing.

**KV cache.** Each layer stores $K, V \in \mathbb{R}^{B \times n_{kv} \times S \times d_h}$, hence
$2 \cdot B \cdot S \cdot L \cdot n_{kv} \cdot d_h \cdot \text{bytes}$. It grows linearly in both batch
and context: Llama-2-7B needs 0.5 MiB per token, so 2.1 GB at 4k and 17 GB at 32k — more than
its 13.5 GB of bf16 weights. Llama-3-8B's GQA (8 KV heads instead of 32) cuts that by 4×.

**FLOPs.** A token passing through `Linear(a, b)` costs one multiply-add per weight: $2ab$ FLOPs.
Backward runs two matmuls of the same size per layer — $\partial L/\partial x = g W$ and
$\partial L/\partial W = g^\top x$ — so $4ab$. Summed over all weights: $2N + 4N = 6N$ per
token. `FlopCounterMode` confirms it exactly on a matmul-only network (with an input that
requires grad; otherwise the first layer's input gradient is skipped and you get
$6N - 2 n_{\text{first}}$ per token).

What $6ND$ ignores:
- **Attention scores** $QK^\top$ and $AV$ have no parameters: $\approx 12\,L\,d\,s$ extra FLOPs per
  token for training at context $s$ (PaLM, Appendix B), ~16% for Llama-2-7B at 4k and dominant
  at very long context.
- The **embedding lookup** is a gather, not a matmul, so counting its $Vd$ parameters slightly
  overestimates; the (untied) LM head *is* a matmul and is counted correctly.
- Norms, activations, softmax and the optimizer update are $O(N)$ or $O(d)$ per token — small.
- **Activation recomputation** (checkpointing) re-runs the forward: hardware does ~$8ND$,
  while MFU is conventionally reported on the $6ND$ "model FLOPs".

### Complexity

All four functions are $O(1)$ arithmetic — the point is getting the constants right.

### Common bugs

- Giving `k_proj`/`v_proj` `d_model` outputs under GQA (overcounts Llama-3-8B by 0.8B).
- Counting the LM head when embeddings are tied — or forgetting it when they are not.
- LayerNorm counted as `d` (it has a bias), forgetting the final norm, or only one norm per layer.
- Adding a bias to the LM head, or forgetting `bias=True` applies to *every* projection.
- Memory: Adam at 4 B/param (it keeps two fp32 states), forgetting the fp32 master weights in
  mixed precision (12 B instead of 16), or mixing GiB ($2^{30}$) and GB ($10^9$) — 7% apart.
- KV cache without the factor 2 for K *and* V, or with `n_heads` instead of `n_kv_heads`.
- FLOPs as $2ND$ (forward only) or $3ND$ (counting multiply-adds instead of FLOPs).

### Interview follow-ups

1. **Where does activation memory come from, and what does checkpointing buy?** Every layer
   saves its inputs and intermediates for backward: roughly $34\,s\,b\,d$ bytes per layer in
   16-bit for a GPT-style block, plus $5\,a\,s^2 b$ for materialised attention scores, which
   FlashAttention removes (Korthikanti et al., 2022). For Llama-2-7B at $s = 4096$, $b = 1$
   that is ~0.6 GB per layer, ~18 GB in total. Full checkpointing keeps only each layer's
   input ($2sbd$ bytes, ~34 MB per layer) and recomputes the rest during backward — one extra
   forward pass, i.e. ~33% more compute (8ND instead of 6ND).
2. **What are the ZeRO stages' per-GPU memory formulas?** With $\Psi$ parameters,
   $K = 12$ bytes of fp32 optimizer state and $N_d$ data-parallel ranks: DDP $(2+2+K)\Psi$;
   ZeRO-1 shards the optimizer, $4\Psi + K\Psi/N_d$; ZeRO-2 also shards gradients,
   $2\Psi + (2+K)\Psi/N_d$; ZeRO-3/FSDP also shards parameters, $(2+2+K)\Psi/N_d$. For
   Llama-2-7B on 8 GPUs: 108 → 37 → 25 → 13.5 GB, so "8×80 GB" works from ZeRO-1 up
   (activations permitting), while plain DDP does not fit at all.
3. **What is MFU and what is a good number?** Model FLOPs utilisation = achieved model FLOP/s
   ($6N$ + attention, per token × tokens/s) ÷ hardware peak. Llama-2-7B's reported 184k
   A100-hours for 2T tokens implies ~39%; well-tuned dense LLM training typically lands at
   35–55%. Communication, pipeline bubbles, memory-bound kernels and recomputation eat the rest.
4. **Why is LLM inference memory-bandwidth bound at small batch?** One decode step reads every
   weight once ($2N$ bytes in bf16) and does $2N$ FLOPs per sequence — an arithmetic intensity
   of about $B$ FLOPs/byte for batch $B$. An A100 can do ~312 TFLOP/s but stream only ~2 TB/s,
   so it needs $B \gtrsim 150$ to become compute-bound; at batch 1, Llama-2-7B is limited to
   roughly 13.5 GB ÷ 2 TB/s ≈ 7 ms per token. Hence batching, weight quantization, and
   GQA/MQA to shrink the KV cache that is also read every step.
5. **How long would pre-training take?** $C = 6ND$; time $= C / (\#\text{GPUs} \cdot \text{peak} \cdot \text{MFU})$.
   Chinchilla-optimal training uses $D \approx 20N$ tokens, so $C \approx 120N^2$: a 70B model on
   1.4T tokens is $5.9\times10^{23}$ FLOPs ≈ 53 days on 1024 A100s at 40% MFU.
6. **Where else does memory go in practice?** Temporary buffers (all-reduce buckets, gathered
   FSDP parameters, logits $B \cdot S \cdot V$ in fp32 — 0.5 GB for one 4k sequence with a 32k
   vocabulary), allocator fragmentation, and CUDA context/workspaces. Leave 10–20% headroom.